## Assignment 7: Multi-Model AI Systems

### Part 1: Model Size Selection and Benchmarking

#### 1.1 Define Model Size Categories
* Ultra-Light: < 1B
* Small: 1 - 3 B
* Medium: 3 - 7B
* Large: 8 - 9B
* X-Large: 10B +

#### 1.2 Define Prompt Types
* Classification: Determine whether a listing belongs in the service, product offering or fundraising category.
* Prediction: Identify if a listing is safe and does not violate any university policies.
* Summarize: Turn a product/service description into a tagline for the search page.
* Text Generation: Help an interested customer start a conversation with the seller.
* Structured Extraction: Get specific data about Price, Service, and Location from a listing.


#### 1.3 Create Evaluation Prompts
##### Classification
1. ‘Hello! The Illini Service Dogs are having a fundraiser bake sale today.’
2. ‘Offering math tutoring services for 20 dollars an hour.’
3. ‘Selling a used air fryer for 30 dollars’
4. ‘Car wash event to raise money for a student organization.’
5. ‘Freelance photography, graduation photoshoot available.’
##### Prediction
1. “Does this violate policy? Offering essay writing services for payment.”
2. “Determine if safe: Selling used textbooks.”
3. “Is this listing appropriate? Providing tutoring help for exams.”
4. “Check policy compliance: Selling stolen electronics.”
5. “Does this follow student code of conduct: Student representatives must be elected by majority.”

##### Summarization
1. Summarize into a short tagline: ‘Used bike in good condition, recently repaired, perfect for campus travel.’
2. Create a one-line summary: ' Affordable tutoring services for math and science courses.’
3. Summarize this listing: ‘Mini fridge with freezer, lightly used, great for dorm rooms.’
4. Write a short tagline: ‘Student organization fundraiser selling homemade cookies.’
5. Summarize: ‘Graphic design services for resumes, flyers, and social media.’

##### Text Generation
1. “Write a message to the seller asking if the item is still available.”

2. “Generate a polite inquiry about price negotiation.”
3. “Write a message asking for more details about the service.”
4. “Generate a friendly message expressing interest in buying.”
5. “Write a short message scheduling a meetup.”

##### Structured Extraction
1. “Extract fields: ‘Mini fridge, $60, Urbana campus.’”
2. “Convert to structured format: ‘Graphic design service, 15 dollars per project, remote.’”
3. “Extract {item, price}: ‘Desk chair for $30.’”
4. “Convert to structured format: ‘Bake sale at the Quad, cookies 2 dollars each.’”
5. “Extract fields: ‘Tutoring, $15, Urbana campus.’”




#### 1.4 Define Evaluation Metrics
##### Instruction Adherence
Did the Model follow instructions? This includes staying within the requested task (example classify, summarize, extract). It should also avoid unnecessary information.

##### Accuracy
Was the answer correct? Did it include a logical output with the response reflecting some input listings correct classification, prediction, or extracted value.

##### Clarity
Was the output understandable? This includes having clear and concise language without confusing phrasing.

##### Structure
Did the output follow the expected format with organized content, clean labels, and readable summaries?

1–3: Poor
4–6: Moderate
7–8: Good
9–10: Excellent

In [1]:
import torch
import time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json
import os

In [ ]:
import torch
import time
import json
import gc
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- 1. CONFIGURATION ---
MODEL_MAP = {
    # "Ultra-Light": [
    #     "Qwen/Qwen2.5-0.5B-Instruct",
    #     "HuggingFaceTB/SmolLM2-360M-Instruct",
    #     "Qwen/Qwen2.5-1.5B-Instruct"
    # ],
    # "Small": [
    #     "Qwen/Qwen2.5-3B-Instruct",
    #     "stabilityai/stablelm-zephyr-3b",
    #     "microsoft/phi-2"
    # ]
    # "Medium": [
    #     "Deci/DeciLM-7B",
    #     "MaziyarPanahi/calme-2.1-phi3.5-4b", ],
        # "frameai/Loxa-4B"],
    # "Large": [
    #     "mistralai/Mistral-7B-v0.3",
    #     "Qwen/Qwen2.5-7B-Instruct",
    #     "tiiuae/falcon-11B"],
    # "X-Large": [
    #     "Qwen/Qwen2.5-14B",
    #     "tiiuae/Falcon3-MoE-2x7B-Instruct",
    #     "mistralai/Mistral-Nemo-12B-Instruct-v1"]

}

prompts_data = [
    {"type": "Classification", "text": "Classify: 'Hello! The Illini Service Dogs are having a fundraiser bake sale today.'"},
    {"type": "Classification", "text": "Classify: 'Offering math tutoring services for $20 an hour.'"},
    {"type": "Classification", "text": "Classify: 'Selling a used air fryer for $30'"},
    {"type": "Classification", "text": "Classify: 'Car wash event to raise money for a student organization.'"},
    {"type": "Classification", "text": "Classify: 'Freelance photography, graduation photoshoot available.'"},
    {"type": "Prediction", "text": "Does this violate policy? Offering essay writing services for payment."},
    {"type": "Prediction", "text": "Determine if safe: Selling used textbooks."},
    {"type": "Prediction", "text": "Is this listing appropriate? Providing tutoring help for exams."},
    {"type": "Prediction", "text": "Check policy compliance: Selling stolen electronics."},
    {"type": "Prediction", "text": "Does this violate university policy? Promoting a local bar crawl."},
    {"type": "Summarize", "text": "Summarize into a short tagline: 'Used bike in good condition, recently repaired, perfect for campus travel.'"},
    {"type": "Summarize", "text": "Create a one-line summary: 'Affordable tutoring services for math and science courses.'"},
    {"type": "Summarize", "text": "Summarize this listing: 'Mini fridge with freezer, lightly used, great for dorm rooms.'"},
    {"type": "Summarize", "text": "Write a short tagline: 'Student organization fundraiser selling homemade cookies.'"},
    {"type": "Summarize", "text": "Summarize: 'Graphic design services for resumes, flyers, and social media.'"},
    {"type": "Text Gen", "text": "Write a message to the seller asking if the item is still available."},
    {"type": "Text Gen", "text": "Generate a polite inquiry about price negotiation."},
    {"type": "Text Gen", "text": "Write a message asking for more details about the service."},
    {"type": "Text Gen", "text": "Generate a friendly message expressing interest in buying."},
    {"type": "Text Gen", "text": "Write a short message scheduling a meetup."},
    {"type": "Extraction", "text": "Extract fields: 'Mini fridge, $60, Urbana campus.'"},
    {"type": "Extraction", "text": "Convert to structured format: 'Graphic design service, $15 per project, remote.'"},
    {"type": "Extraction", "text": "Extract {item, price}: 'Desk chair for $30.'"},
    {"type": "Extraction", "text": "Convert to structured format: 'Bake sale at the Quad, cookies 2$ each.'"},
    {"type": "Extraction", "text": "Extract fields: 'Tutoring, $15, Urbana campus.'"}
]

all_benchmarks = []
# The filename for your JSON output
OUTPUT_FILE = "harbor_benchmarks.json"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Targeting device: {device.upper()}")

# --- 2. EXECUTION LOOP ---
for category, models in MODEL_MAP.items():
    for model_id in models:
        print(f"\n{'='*50}\n>>> PROCESSING: {model_id} ({category})\n{'='*50}")

        try:
            tokenizer = AutoTokenizer.from_pretrained(
                model_id,
                trust_remote_code=True)

            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4"
            )

            model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True,
                low_cpu_mem_usage=True,
                attn_implementation="eager"
            )

            if model.config.pad_token_id is None:
                model.config.pad_token_id = model.config.eos_token_id

            for p in prompts_data:
                inputs = tokenizer(p['text'], return_tensors="pt").to(device)

                start_time = time.perf_counter()
                output_tokens = model.generate(**inputs, max_new_tokens=50, do_sample=False)
                end_time = time.perf_counter()

                latency = end_time - start_time
                new_tokens = len(output_tokens[0]) - len(inputs['input_ids'][0])
                tps = new_tokens / latency if latency > 0 else 0
                response_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

                # 1. Create the result entry
                new_entry = {
                    "Category": category,
                    "Model": model_id,
                    "Task Type": p['type'],
                    "Prompt": p['text'],
                    "Result": response_text,
                    "Latency (sec)": round(latency, 3),
                    "Tokens Per Sec": round(tps, 2)
                }

                # 2. SAFETY SAVE: Load, Append, and Save immediately
                final_data = []
                if os.path.exists(OUTPUT_FILE):
                    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
                        try:
                            final_data = json.load(f)
                        except json.JSONDecodeError:
                            final_data = []

                final_data.append(new_entry)

                with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
                    json.dump(final_data, f, indent=4)

                print(f"  [+] {p['type']} saved. Speed: {tps:.2f} t/s")

        except Exception as e:
            print(f"\n[ERROR] Skipping {model_id} due to failure: {e}")
            continue

        # --- 3. MEMORY FLUSH (GPU RESET) ---
        del model
        del tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
        time.sleep(1.5)

print("\n" + "#"*50)
print(f"BENCHMARK FINISHED. Final JSON path: {os.path.abspath(OUTPUT_FILE)}")
print("#"*50)